# Music Annotation Project V9.0B
## Notebook 60: Clean V8.8 HITL Batch Operationalisation

**Runtime:** CPU. In Google Colab choose **Runtime > Run all**.

This notebook is a clean batch wrapper for the completed V8.8 correction contract. It deliberately does not depend on Notebook 45, 46 or 50 run directories. It consumes only:

1. current Notebook 07 HITL audit records and their resolved aligned pages / corrected masks;
2. one completed Notebook 53R3 registered-replay implementation run;
3. the V8.8 rule `M5_REMOVE = M3 minus SAFE_REMOVE`;
4. Telea radius 3 whole-page correction with exact outside-authority immutability.

### Fail-closed rule
For pages present in the registered V8.8 replay roster, the notebook uses the registered `m3.png`, `safe_remove.png` and `m5.png` masks. For a HITL page not present in that roster, it does **not** invent a SAFE_REMOVE decision. It retains the authoritative corrected mask as M3, sets SAFE_REMOVE to empty, and records `M3_FALLBACK_UNSEEN_PAGE`. This permits all HITL pages to run without silently fabricating learned decisions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow


## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, sys, platform, traceback
import cv2
import numpy as np
import pandas as pd

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
HITL_AUDIT_ROOT=ROOT/'human_in_the_middle'/'audit'
N53_ROOT=ROOT/'observed_print_classifier_v8_8_notebook53R3'
OUTPUT_ROOT=ROOT/'v9_0b_notebook60_v8_8_hitl_batch'
N53_RUN_OVERRIDE=None
PAGE_ALLOWLIST=None
RAISE_AT_END_ON_PAGE_FAILURE=False
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook60_v9_0b'
OUT=OUTPUT_ROOT/RUN
PAGES_OUT=OUT/'pages'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; FAILURES=OUT/'failures'; UPLOAD=OUT/'upload'
for p in [OUT,PAGES_OUT,TABLES,AUDIT,FAILURES,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)


## 2. Helpers and explicit completed-run resolution

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def read_gray(path):
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    if im is None: raise RuntimeError(f'Unreadable grayscale image: {path}')
    return im

def latest_complete(root,required):
    hits=[]
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

REQ53=['stage_f_verdict.json','experiment_identity.json','tables/page_equivalence.csv','pages']
N53=Path(N53_RUN_OVERRIDE) if N53_RUN_OVERRIDE else latest_complete(N53_ROOT,REQ53)
if N53 is None: raise RuntimeError(f'No complete Notebook 53R3 run found below {N53_ROOT}')
v53=json.loads((N53/'stage_f_verdict.json').read_text())
i53=json.loads((N53/'experiment_identity.json').read_text())
if v53.get('verdict')!='PASS_STAGE_F_REGRESSION_COMPATIBILITY': raise RuntimeError(f'Notebook 53R3 verdict is not PASS: {v53.get("verdict")}')
if v53.get('mode')!='REGISTERED_REPLAY' or i53.get('mode')!='REGISTERED_REPLAY': raise RuntimeError('Notebook 53R3 is not the registered replay implementation.')
if int(i53.get('inpaint_radius',-1))!=3: raise RuntimeError('Notebook 53R3 does not declare Telea radius 3.')
print('V8.8 registered source:',N53)


## 3. Resolve Notebook 07 HITL pages from audit records

In [ ]:
def audit_records(root):
    rows=[]; failures=[]
    for p in sorted(root.rglob('*.json')) if root.exists() else []:
        try:
            d=json.loads(p.read_text())
            if not isinstance(d,dict): raise ValueError('audit root is not an object')
            aligned=d.get('aligned_output') or d.get('aligned_page') or d.get('observed_output')
            mask=d.get('corrected_mask_output') or d.get('corrected_mask')
            source=d.get('source') or d.get('source_file')
            if aligned and mask:
                ap=Path(str(aligned)); mp=Path(str(mask))
                page=ap.stem
                if page.endswith('_aligned'): page=page[:-8]
                rows.append({'page':page,'audit_path':str(p),'aligned_path':str(ap),'mask_path':str(mp),'source_path':str(source) if source else None,'source_md5':d.get('source_md5'),'timestamp':d.get('timestamp') or d.get('reviewed_utc') or ''})
        except Exception as e: failures.append({'audit_path':str(p),'error':repr(e)})
    return pd.DataFrame(rows),pd.DataFrame(failures)

roster,parse_failures=audit_records(HITL_AUDIT_ROOT)
parse_failures.to_csv(AUDIT/'audit_parse_failures.csv',index=False)
if roster.empty: raise RuntimeError(f'No Notebook 07 audit records with aligned and corrected-mask outputs found below {HITL_AUDIT_ROOT}')
roster=roster.sort_values(['page','timestamp','audit_path']).drop_duplicates('page',keep='last').reset_index(drop=True)
if PAGE_ALLOWLIST is not None: roster=roster[roster.page.isin(set(PAGE_ALLOWLIST))].copy()
if roster.empty: raise RuntimeError('HITL roster is empty after PAGE_ALLOWLIST.')
roster.to_csv(TABLES/'hitl_roster.csv',index=False)
print('Notebook 07 HITL pages:',len(roster))


## 4. V8.8 mask resolution and correction engine

In [ ]:
def registered_masks(page,shape):
    d=N53/'pages'/page
    files={k:d/f for k,f in {'m3':'m3.png','safe_remove':'safe_remove.png','m5':'m5.png'}.items()}
    if all(p.exists() for p in files.values()):
        out={k:(read_gray(p)>0) for k,p in files.items()}
        if any(v.shape!=shape for v in out.values()): raise RuntimeError(f'{page}: registered replay mask shape mismatch')
        expected=out['m3'] & ~out['safe_remove']
        if np.logical_xor(expected,out['m5']).any(): raise RuntimeError(f'{page}: registered M5 != M3 minus SAFE_REMOVE')
        return out,'REGISTERED_REPLAY'
    return None,'M3_FALLBACK_UNSEEN_PAGE'

def telea_components(observed,mask,radius=3,margin=48):
    result=observed.copy(); n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8); h,w=mask.shape
    boxes=[]
    for i in range(1,n):
        x,y,bw,bh,a=stats[i]
        if a>=8: boxes.append((max(0,x-margin),max(0,y-margin),min(w,x+bw+margin),min(h,y+bh+margin)))
    for x0,y0,x1,y1 in boxes:
        local_m=(mask[y0:y1,x0:x1].astype(np.uint8)*255); local_i=observed[y0:y1,x0:x1]
        cand=cv2.inpaint(local_i,local_m,radius,cv2.INPAINT_TELEA); patch=result[y0:y1,x0:x1]; patch[local_m>0]=cand[local_m>0]
    out=observed.copy(); out[mask]=result[mask]
    if not np.array_equal(out[~mask],observed[~mask]): raise RuntimeError('outside-authority drift')
    return out


## 5. Process the entire HITL roster

In [ ]:
results=[]
for _,r in roster.iterrows():
    page=str(r.page); page_dir=PAGES_OUT/page
    try:
        observed=read_gray(Path(r.aligned_path)); seed=read_gray(Path(r.mask_path))
        if seed.shape!=observed.shape: raise RuntimeError(f'shape mismatch observed={observed.shape} mask={seed.shape}')
        authority=seed>0
        registered,mode=registered_masks(page,observed.shape)
        if registered is None:
            m3=authority.copy(); safe_remove=np.zeros_like(authority); m5=m3.copy()
        else:
            m3=registered['m3']; safe_remove=registered['safe_remove']; m5=registered['m5']
            if np.any(m3 & ~authority): raise RuntimeError('registered M3 expands beyond current authoritative mask')
            if np.any(safe_remove & ~m3): raise RuntimeError('SAFE_REMOVE outside M3')
        corrected=telea_components(observed,m5,radius=3,margin=48)
        page_dir.mkdir(parents=True,exist_ok=False)
        cv2.imwrite(str(page_dir/'observed.png'),observed)
        cv2.imwrite(str(page_dir/'authoritative_mask.png'),authority.astype(np.uint8)*255)
        cv2.imwrite(str(page_dir/'m3.png'),m3.astype(np.uint8)*255)
        cv2.imwrite(str(page_dir/'safe_remove.png'),safe_remove.astype(np.uint8)*255)
        cv2.imwrite(str(page_dir/'m5.png'),m5.astype(np.uint8)*255)
        cv2.imwrite(str(page_dir/'corrected.png'),corrected)
        diff=np.zeros_like(observed); diff[corrected!=observed]=255; cv2.imwrite(str(page_dir/'difference.png'),diff)
        row={'page':page,'status':'PASS','mode':mode,'aligned_path':r.aligned_path,'mask_path':r.mask_path,'authority_pixels':int(authority.sum()),'m3_pixels':int(m3.sum()),'safe_remove_pixels':int(safe_remove.sum()),'m5_pixels':int(m5.sum()),'changed_pixels':int((corrected!=observed).sum()),'outside_authority_drift':int(np.sum(corrected[~authority]!=observed[~authority])),'m5_formula_xor':int(np.logical_xor(m5,m3&~safe_remove).sum()),'observed_semantic_sha256':array_hash(observed),'authority_semantic_sha256':array_hash(authority.astype(np.uint8)),'m5_semantic_sha256':array_hash(m5.astype(np.uint8)),'corrected_semantic_sha256':array_hash(corrected),'error':''}
        atomic_json(page_dir/'page_manifest.json',row); results.append(row); print('PASS',page,mode)
    except Exception as e:
        err={'page':page,'status':'FAIL','mode':'FAILED','aligned_path':r.aligned_path,'mask_path':r.mask_path,'error':repr(e),'traceback':traceback.format_exc()}
        atomic_json(FAILURES/f'{page}.json',err); results.append(err); print('FAIL',page,repr(e))

result_df=pd.DataFrame(results); result_df.to_csv(TABLES/'page_results.csv',index=False)


## 6. Batch verdict and final upload package

In [ ]:
passed=result_df[result_df.status=='PASS']; failed=result_df[result_df.status=='FAIL']
registered=int((passed.get('mode',pd.Series(dtype=str))=='REGISTERED_REPLAY').sum())
fallback=int((passed.get('mode',pd.Series(dtype=str))=='M3_FALLBACK_UNSEEN_PAGE').sum())
checks={'pages_discovered':len(roster)>0,'all_pages_accounted_for':len(result_df)==len(roster),'zero_page_failures':len(failed)==0,'zero_outside_authority_drift':bool(len(passed) and passed.outside_authority_drift.fillna(0).eq(0).all()),'zero_m5_formula_xor':bool(len(passed) and passed.m5_formula_xor.fillna(0).eq(0).all())}
verdict='PASS_ALL_HITL_PAGES_PROCESSED' if all(checks.values()) else 'PARTIAL_HITL_BATCH_WITH_RECORDED_FAILURES'
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0B-notebook60-clean-v8.8-hitl-batch','verdict':verdict,'checks':checks,'hitl_pages':len(roster),'passed_pages':len(passed),'failed_pages':len(failed),'registered_replay_pages':registered,'m3_fallback_unseen_pages':fallback,'notebook53_source':str(N53),'notebook53_identity_sha256':sha256_file(N53/'experiment_identity.json'),'safe_protect_runtime_enabled':False,'telea_radius':3,'notebook45_dependency':False,'notebook46_dependency':False,'notebook50_dependency':False}
atomic_json(OUT/'notebook60_verdict.json',summary)
lines=['NOTEBOOK 60 / V9.0B CLEAN V8.8 HITL BATCH','',f'Verdict: {verdict}',f'HITL pages: {len(roster)}',f'Passed: {len(passed)}',f'Failed: {len(failed)}',f'Registered replay: {registered}',f'M3 fallback unseen pages: {fallback}',f'Notebook 53R3 source: {N53}','','No dependency on Notebook 45, 46 or 50 run directories.','Unseen pages fail closed to exact M3 treatment; SAFE_REMOVE is never invented.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
for src in [OUT/'executive_summary.txt',OUT/'notebook60_verdict.json',TABLES/'hitl_roster.csv',TABLES/'page_results.csv',AUDIT/'audit_parse_failures.csv']:
    if src.exists(): shutil.copy2(src,UPLOAD/src.name)
zip_path=UPLOAD/'notebook60_v9_0b_review_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.iterdir()):
        if p.is_file() and p!=zip_path: z.write(p,p.name)
print(chr(10).join(lines)); print('Review bundle:',zip_path)
if len(failed) and RAISE_AT_END_ON_PAGE_FAILURE: raise RuntimeError(f'{len(failed)} page(s) failed; evidence was written before stopping.')
